# Lab 12: Geometric Transformations — 🎓 INSTRUCTOR VERSION> **Do not distribute to students.****Complete solutions for all three activities.**---### ⏱️ Time Guide (90 min)| Section | Time | Notes ||---------|------|-------|| Setup + SIFT/matching | 15 min | Run detection example || Homography + validation | 10 min | Understand RANSAC || Activity 1: Pose correction | 20 min | Core challenge || Activity 2: Augmented reality | 20 min | More complex || Activity 3: Panorama | 20 min | Inverse homography || Discussion + reflection | 5 min | Connect concepts |---

# Lab 12: Position Models as Geometric Transformations**Computer Vision Course**Building on corner detection and matching (Lab 11), today you'll learn how to estimate **geometric transformations** between images and use them for powerful applications!**What you'll learn:**- Homography estimation from matched keypoints- SIFT feature detection and matching- RANSAC for robust estimation- Perspective transformation using cv2.warpPerspective**What you'll build:**- Pose correction (dewarp distorted images)- Augmented reality (replace objects in scenes)- Panorama stitching (combine overlapping images)**Why this matters:**Geometric transformations are everywhere in computer vision:- Document scanning and correction- Augmented Reality applications- Panoramic photography- 3D reconstruction- Robot navigation**Connection to previous labs:**- Lab 11: Corner detection and patch matching- Lab 12 (today): Estimate transformations from matches- Real applications: Image stitching, AR, 3D reconstruction!

## Setup

In [ ]:
"""Computer Vision Course - Lab 12: Geometric TransformationsThis cell sets up the environment.Works automatically for both local and Google Colab!"""import osimport sys# Detect environmentIN_COLAB = 'google.colab' in sys.modulesprint("=" * 60)print("Computer Vision - Lab 12: Geometric Transformations")print("=" * 60)if IN_COLAB:    print("\n🔵 Running on Google Colab")    print("-" * 60)        if not os.path.exists('computer-vision'):        print("📥 Cloning repository...")        !git clone https://github.com/mjck/computer-vision.git        print("✓ Repository cloned successfully")    else:        print("✓ Repository already exists")        %cd computer-vision/labs/lab12_transformations    print(f"✓ Current directory: {os.getcwd()}")        sys.path.insert(0, '/content/computer-vision')    print("✓ Python path configured")        print("-" * 60)    print("🟢 Colab setup complete!\n")    else:    print("\n🟢 Running locally")    print("-" * 60)    print(f"✓ Current directory: {os.getcwd()}")        repo_root = os.path.abspath('../..')    if repo_root not in sys.path:        sys.path.insert(0, repo_root)    print(f"✓ Repository root: {repo_root}")        print("-" * 60)    print("🟢 Local setup complete!\n")print("=" * 60)print("✅ Environment ready!")print("=" * 60)

## Import Libraries

In [ ]:
import numpy as npimport cv2# Import course utilitiestry:    from sdx import cv_imread, cv_imshow, cv_grayread    print("✓ sdx module loaded")except ImportError as e:    print(f"❌ Could not import sdx: {e}")    raiseprint("✓ All imports successful")

---## Choosing an Image PairWe have 4 different scenarios to explore:1. **`detection`** — Simple example to understand the workflow2. **`correction`** — For Activity 1 (pose correction)3. **`augmented`** — For Activity 2 (augmented reality)4. **`panorama`** — For Activity 3 (panorama stitching)Start with `detection` to learn the pipeline, then switch for activities!

In [ ]:
# Choose your scenarioSOURCE = 'detection'  # Options: 'detection', 'correction', 'augmented', 'panorama'print(f"✓ Selected scenario: {SOURCE}")print("\nNote: You'll change this for each activity later!")

---## Loading Images### TerminologyWe use OpenCV's keypoint framework terminology:- **Train image:** The "object" we're looking for (reference)- **Query image:** The "scene" where we search for the objectThis naming matches OpenCV's SIFT/matching documentation.

### Load Train Image (Object)

In [ ]:
train = cv_grayread(f'images/{SOURCE}-train.png')train_height, train_width = train.shapeprint(f"Train image: {train.shape}")cv_imshow(train)print(f"This is the 'object' we'll search for")

### Load Query Image (Scene)

In [ ]:
query = cv_grayread(f'images/{SOURCE}-query.png')query_height, query_width = query.shapeprint(f"Query image: {query.shape}")cv_imshow(query)print(f"This is the 'scene' where we'll find the object")

---## Extracting Keypoints and Descriptors**SIFT (Scale-Invariant Feature Transform):**- Detects keypoints at multiple scales (Lab 9 concepts!)- Computes rotation-invariant descriptors- Each descriptor is a 128-dimensional vector- Much better than simple patch matching (Lab 11)**What SIFT does:**1. Find keypoints (corners) across scale-space2. Assign dominant orientation to each keypoint3. Compute descriptor in normalized patch**Why SIFT is powerful:**- ✅ Scale-invariant (works at different sizes)- ✅ Rotation-invariant (works at different angles)- ✅ Illumination-robust (handles lighting changes)- ✅ Distinctive (128-dim vectors are unique)

In [ ]:
# Create SIFT detectorsift = cv2.SIFT_create()# Detect keypoints and compute descriptors# None = process entire image (no mask)train_keypoints, train_descriptors = sift.detectAndCompute(train, None)query_keypoints, query_descriptors = sift.detectAndCompute(query, None)print(f"Train: {len(train_keypoints)} keypoints detected")print(f"Query: {len(query_keypoints)} keypoints detected")print(f"\nDescriptor dimensionality: {train_descriptors.shape[1]}")print("(Each keypoint described by a 128-dimensional vector)")

---## Keypoint Matching Based on Descriptor SimilarityInstead of comparing patches (Lab 11), we compare **SIFT descriptors**.**FLANN (Fast Library for Approximate Nearest Neighbors):**- Efficient algorithm for finding similar descriptors- Much faster than brute-force search- Used for large-scale matching**k-NN Matching (k=2):**- For each query descriptor, find 2 nearest train descriptors- Why 2? To apply Lowe's ratio test (next step)

In [ ]:
# Create FLANN matchermatcher = cv2.FlannBasedMatcher()# For each query descriptor, find k=2 nearest matchesmatches = matcher.knnMatch(query_descriptors, train_descriptors, k=2)print(f"Found {len(matches)} query keypoints")print("Each has 2 candidate matches from train image")

### Lowe's Ratio Test**The problem:** Many matches are ambiguous or wrong!**Lowe's ratio test (from SIFT paper):**- Look at the two best matches for each query descriptor- If **best_distance < 0.8 × second_best_distance**, accept the match- Otherwise, reject both (ambiguous match)**Why it works:**- Good matches: nearest neighbor much closer than second nearest- Bad matches: both neighbors similarly distant (ambiguous)- Threshold 0.8 is empirically optimal (from SIFT paper)

In [ ]:
# Binary mask for visualization# (which matches to draw)matches_mask = []# List of good matches (passed ratio test)good_matches = []# Each element is (first_match, second_match)for first, second in matches:    # Ratio test: is first significantly better than second?    if first.distance < 0.8 * second.distance:        # Good match! Keep it        matches_mask.append((1, 0))  # Draw first, not second        good_matches.append(first)    else:        # Ambiguous match, reject both        matches_mask.append((0, 0))  # Draw neitherprint(f"Total matches: {len(matches)}")print(f"Good matches (passed ratio test): {len(good_matches)}")print(f"Rejected (ambiguous): {len(matches) - len(good_matches)}")print(f"\nAcceptance rate: {100 * len(good_matches) / len(matches):.1f}%")

### Visualize Matches

In [ ]:
# Draw matches (only good ones, due to matches_mask)matches_output = cv2.drawMatchesKnn(    query, query_keypoints,      # Query image and keypoints    train, train_keypoints,      # Train image and keypoints      matches,                     # All matches    None,                        # Output image (None = create new)    matchesMask=matches_mask     # Which matches to draw)cv_imshow(matches_output)print("Green lines connect matched keypoints")print("Only matches that passed Lowe's ratio test are shown")

---## Estimating Pose (Homography)**The goal:** Find the geometric transformation between train and query images.**Homography (H):**- A 3×3 matrix that maps points from one image to another- Represents perspective transformation- Can handle rotation, scale, translation, and perspective distortion**What homography does:**```[x']     [h11 h12 h13]   [x][y']  =  [h21 h22 h23] × [y][1 ]     [h31 h32 h33]   [1]```Then: x_final = x'/1, y_final = y'/1 (homogeneous coordinates)**Use case:** Maps train image coordinates → query image coordinates

In [ ]:
# Prepare point correspondences from good matchespose_src = []  # Train image pointspose_dst = []  # Query image pointsfor match in good_matches:    # trainIdx: index in train_keypoints    # queryIdx: index in query_keypoints    pose_src.append(train_keypoints[match.trainIdx].pt)    pose_dst.append(query_keypoints[match.queryIdx].pt)# Convert to NumPy arrays (required by findHomography)pose_src = np.array(pose_src)pose_dst = np.array(pose_dst)print(f"Using {len(pose_src)} point correspondences")print(f"Source points shape: {pose_src.shape}")print(f"Destination points shape: {pose_dst.shape}")

### RANSAC for Robust Estimation**The problem:** Some matches are still wrong (outliers)!**RANSAC (Random Sample Consensus):**1. Randomly sample small subset of matches2. Fit homography to this subset3. Count how many other matches agree (inliers)4. Repeat many times5. Keep homography with most inliers**Why it works:**- Outliers are random → unlikely to be in same random sample- Correct matches (inliers) → consistent with same homography- Robust to even 50% outliers!**cv2.findHomography with RANSAC:**- Automatically applies RANSAC- Returns H (homography) and mask (inlier/outlier labels)

In [ ]:
# Estimate homography using RANSACH, mask = cv2.findHomography(    pose_src,      # Source points (train image)    pose_dst,      # Destination points (query image)    cv2.RANSAC     # Use RANSAC for robustness)# Count inliersinliers = np.sum(mask)outliers = len(mask) - inliersprint(f"Homography estimated:")print(H)print(f"\nRANSAC results:")print(f"  Inliers: {inliers} ({100*inliers/len(mask):.1f}%)")print(f"  Outliers: {outliers} ({100*outliers/len(mask):.1f}%)")print("\n✓ Homography is robust even with outliers!")

---## Validating the HomographyTo visualize the transformation, we'll:1. Define a rectangle around the train image2. Transform it using the homography3. Draw the transformed rectangle on the query imageThis shows where the object appears in the scene!

In [ ]:
# Define rectangle around train image# Note: OpenCV uses (x, y) ordering!rectangle_src = [    (0, 0),                          # Top-left    (0, train_height),               # Bottom-left    (train_width, train_height),     # Bottom-right    (train_width, 0),                # Top-right]# Convert to NumPy array with required shape# perspectiveTransform needs shape: (N, 1, 2) with float32rectangle_src = np.array([rectangle_src], dtype=np.float32)print(f"Rectangle corners in train image:")print(rectangle_src.reshape(4, 2))

### Transform Rectangle

In [ ]:
# Apply homography to rectanglerectangle_dst = cv2.perspectiveTransform(rectangle_src, H)print(f"\nRectangle corners in query image (after transformation):")print(rectangle_dst.reshape(4, 2))print("\nThese are where the train image corners appear in the query!")

### Draw Transformed Rectangle

In [ ]:
# Convert to BGR for colored drawingrectangle_output = cv2.cvtColor(query, cv2.COLOR_GRAY2BGR)# Draw lines between consecutive cornerscolor = (0, 255, 0)  # Greenthickness = 2for i in range(4):    # Draw line from corner i to corner (i+1) mod 4    pt1 = tuple(rectangle_dst[0][i].astype(int))    pt2 = tuple(rectangle_dst[0][(i + 1) % 4].astype(int))    cv2.line(rectangle_output, pt1, pt2, color, thickness)cv_imshow(rectangle_output)print("\n✓ Green rectangle shows where train image appears in query!")print("✓ Notice: can handle rotation, scaling, perspective!")

---## Loading Color VersionsFor the activities, we'll work with color images for better visual results.

In [ ]:
# Load color versionscolor_train = cv_imread(f'images/{SOURCE}-train.png')color_query = cv_imread(f'images/{SOURCE}-query.png')print("Color train image:")cv_imshow(color_train)print("\nColor query image:")cv_imshow(color_query)print("\n✓ Color images loaded for activities")

---## Activity 1: Pose Correction**Goal:** Remove perspective distortion from a document or sign.**The task:**1. Switch `SOURCE` to `'correction'`2. Run the entire notebook up to this point3. Write code to **correct the perspective distortion****Hint:** You want to transform the **query image** to look like the **train image**.- Train image: undistorted (frontal view)- Query image: distorted (perspective view)- Need: inverse transformation!**Expected output:** [Reference image](https://drive.google.com/file/d/1TfuBFtbgcpvvvxWUWQzDMaCOqG3RCcPN/view?usp=drive_link)**Think about:**- What transformation do you need?- What size should the output be?- Which cv2 function applies transformations?

In [ ]:
# ── Activity 1: Your code here ──────────────────────────────────────────────output1 = color_train.copy()  # Replace this line with your code# ────────────────────────────────────────────────────────────────────────────cv_imshow(output1)print("Activity 1: Pose correction")

---## 🔑 SOLUTION — Activity 1: Pose Correction### Complete Solution```python# What we want is to invert the effect of the homography.# This can be achieved by literally inverting the matrix.H1 = np.linalg.inv(H)# Parameter 1: the image you want to transform.## Parameter 2: the homography you want to apply.## Parameter 3: the size of the output image. The#              crop is from (0, 0) to this size.train_size = (train_width, train_height)output1 = cv.warpPerspective(color_query, H1, train_size)cv_imshow(output1)```### Explanation**The key insight:** We want the INVERSE transformation!- H maps: train → query- We want: query → train- Solution: H_inv = H⁻¹**Steps:**1. Invert the homography matrix2. Apply to query image3. Output size = train image size### Common Mistakes**Mistake 1:** Using H instead of H⁻¹```pythonoutput1 = cv2.warpPerspective(color_query, H, train_size)  # Wrong direction!```This would try to map query → query (doesn't unwarp)**Mistake 2:** Wrong output size```pythonoutput1 = cv2.warpPerspective(color_query, H1, (query_width, query_height))```Should be train size, not query size!**Mistake 3:** Using wrong image```pythonoutput1 = cv2.warpPerspective(color_train, H1, train_size)```We're correcting the query (distorted) image, not train!### Expected Result- Undistorted, frontal view of the document/sign- Should match the appearance of the train image- No perspective distortion### Teaching TipDraw on board:```Train (frontal) --H--> Query (distorted)                  inverse:         Query (distorted) --H⁻¹--> Corrected (frontal)```Point out: This is how document scanners work!

---## Activity 2: Augmented Reality**Goal:** Replace the object in the scene with a different image.**Setup:** First, load the replacement image.

In [ ]:
# Load the image we'll insertcolor_other = cv_imread('images/augmented-other.png')other_height, other_width, _ = color_other.shapeprint(f"Replacement image: {color_other.shape}")cv_imshow(color_other)print("This will replace the object in the scene")

**The task:**1. Switch `SOURCE` to `'augmented'`2. Run the entire notebook up to this point3. Write code to **replace the object in the query image with the other image****Expected output:** [Reference image](https://drive.google.com/file/d/1l8OMcA73azrcMxQS7lzaSLlOOCY3ouEQ/view?usp=drive_link)**Think about:**- What transformation maps `other` → `query`?- How to avoid overwriting the background?- Hint: cv2.warpPerspective can take an output image parameter!

---## 🔑 SOLUTION — Activity 2: Augmented Reality### Complete Solution```pythonoutput2 = color_query.copy()# Since the objects have very similar sizes, we# could simply use the original homography H and# the result would be reasonably good. However,# for completeness, this code assumes that the# two objects have considerably different sizes.# First, build a rectangle around the new object.rectangle_rev = [    (0, 0),    (0, other_height),    (other_width, other_height),    (other_width, 0),]rectangle_rev = np.array([rectangle_rev], dtype=float)# Find an homography that maps this rectangle# to the warped rectangle around the object.H2, _ = cv.findHomography(rectangle_rev, rectangle_dst, cv.RANSAC)# Here comes the tricky part: warpPerspective# can do the entire job if you manage to find# the correct, most up-to-date documentation.## Parameter 1: the image you want to transform.## Parameter 2: the homography you want to apply.## Parameter 3: the size of the output image. The#              crop is from (0, 0) to this size.## Parameter 4: the background where you want to#              overlay the transformed image.#              (must be consistent with previous#              parameter or will fail silently)## Parameter 5: what do you want to do with the#              points outside the limits of the#              original image. The default is to#              fill them with a solid color.query_size = (query_width, query_height)cv.warpPerspective(color_other, H2, query_size, output2, borderMode=cv.BORDER_TRANSPARENT)cv_imshow(output2)```### Full Solution (Complete from Answers)The answers notebook shows the complete implementation. Key steps:1. **Build rectangle around `other` image**2. **Compute homography:** other → train coordinates3. **Combine with H:** train → query coordinates  4. **Warp `other` directly into query****Simplified approach (if sizes similar):**```pythonoutput2 = color_query.copy()other_size = (query_width, query_height)cv2.warpPerspective(color_other, H, other_size, output2,                     borderMode=cv2.BORDER_TRANSPARENT)```### Explanation**Goal:** Replace train object in query with other image**Challenge:** `other` and `train` have different sizes!**Solution path:**1. H maps train → query2. Need: other → query3. Build: other → train (scale/translate)4. Combine: (other → train) ∘ (train → query) = (other → query)**Alternative (simpler):** If `other` ≈ `train` size, just use H directly### Common Mistakes**Mistake 1:** Overwriting background```pythonoutput2 = cv2.warpPerspective(color_other, H, query_size)  # Loses query background!```Need to warp INTO existing query image**Mistake 2:** Wrong transformation direction```pythonH2_inv = np.linalg.inv(H2)  # Unnecessary inversion```**Mistake 3:** Size mismatchNot accounting for different sizes of `other` vs `train`### Expected Result- `color_other` appears where train object was- Matches perspective of query scene- Background preserved### Teaching Moment"This is how Snapchat/Instagram filters work!"- Detect face features (like our SIFT matching)- Estimate pose (like our homography)- Warp overlay onto face (like our Activity 2)

In [ ]:
# ── Activity 2: Your code here ──────────────────────────────────────────────output2 = color_query.copy()# Write your code here# ────────────────────────────────────────────────────────────────────────────cv_imshow(output2)print("Activity 2: Augmented reality")

---## Activity 3: Panorama Stitching**Goal:** Combine two overlapping images into a panorama.**The task:**1. Switch `SOURCE` to `'panorama'`2. Run the entire notebook up to this point3. Write code to **stitch the train and query images together****Expected output:** [Reference image](https://drive.google.com/file/d/1di7bVdMf67LthIU4fW6-6r4_S88Z5B-x/view?usp=drive_link)**Think about:**- Query image is already placed in the output- Where should train image go?- What transformation aligns train to query?- Need: inverse homography!

In [ ]:
# ── Activity 3: Your code here ──────────────────────────────────────────────# Create large output canvasoutput3 = np.zeros((round(1.3 * query_height), round(1.4 * query_width), 3), dtype=np.uint8)# Place query image in bottom-rightoutput3[-query_height:, -query_width:, :] = color_query# Write your code here to add the train image# ────────────────────────────────────────────────────────────────────────────cv_imshow(output3)print("Activity 3: Panorama stitching")

---## 🔑 SOLUTION — Activity 3: Panorama Stitching### Complete Solution```pythonoutput3 = np.zeros((round(1.3 * query_height), round(1.4 * query_width), 3), dtype=np.uint8)output3[-query_height:, -query_width:, :] = color_query# First, build a rectangle that is a shifted# version of the warped rectangle. Since the# original scene was pasted in the lower right# corner of the expanded scene, the shift is# simply the difference of the dimensions.height, width, _ = output3.shaperectangle_mov = rectangle_dst.copy()rectangle_mov[:, :1] += width - query_widthrectangle_mov[:, 1:] += height - query_height# Find an homography that maps the original# object rectangle to the shifted rectangle.H3, _ = cv.findHomography(rectangle_src, rectangle_mov, cv.RANSAC)# This is the same as we did in Activity 2.size = (width, height)cv.warpPerspective(color_train, H3, size, output3, borderMode=cv.BORDER_TRANSPARENT)cv_imshow(output3)```### Full Solution (Complete from Answers)From the answers notebook:1. **Shift the rectangle_dst** by canvas offset2. **Compute reverse homography** (query → train coords)3. **Warp train image** into the canvas**Key insight:** Query already placed → warp train to align### Explanation**Setup:**- Canvas size: 130% × 140% of query- Query placed at bottom-right**Goal:** Add train image in the correct position**Steps:**1. Compute where train should go (inverse of H)2. Account for query offset in canvas3. Warp train into canvas**The math:**- H maps: train → query coordinates- H_inv maps: query → train coordinates- Need: train → canvas coordinates- Solution: Apply offset, then warp### Common Mistakes**Mistake 1:** Forgetting the offset```pythoncv2.warpPerspective(color_train, H_inv, canvas_size, output3)```Doesn't account for query being in bottom-right!**Mistake 2:** Using H instead of H_inv```pythoncv2.warpPerspective(color_train, H, ...)  # Maps train→query, not train→canvas```**Mistake 3:** Wrong canvas size```pythoncv2.warpPerspective(..., (train_width, train_height), ...)```Should be full canvas size!### Expected Result- Both images visible- Overlap region aligned- No seams in matching areas- Forms complete panorama### Teaching Moment"Professional panorama software does this automatically!"- OpenCV has `cv2.Stitcher` class- Can handle multiple images- Adds blending for seamless transitionsShow: ```pythonstitcher = cv2.Stitcher_create()status, panorama = stitcher.stitch([img1, img2, img3])```

---## Summary### What You Learned1. **SIFT Features**   - Scale and rotation invariant   - 128-dimensional descriptors   - Much better than simple patches2. **Feature Matching**   - FLANN for efficient nearest neighbor search   - Lowe's ratio test to filter ambiguous matches   - Typically accept ~30-50% of initial matches3. **Homography Estimation**   - 3×3 matrix for perspective transformation   - Estimated from point correspondences   - RANSAC for robust estimation (handles outliers)4. **Transformation Applications**   - Pose correction: remove perspective distortion   - Augmented reality: replace objects in scenes   - Panorama stitching: combine overlapping images### Key Functions**SIFT:**```pythonsift = cv2.SIFT_create()keypoints, descriptors = sift.detectAndCompute(image, None)```**Matching:**```pythonmatcher = cv2.FlannBasedMatcher()matches = matcher.knnMatch(desc1, desc2, k=2)```**Homography:**```pythonH, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC)```**Warping:**```pythonoutput = cv2.warpPerspective(image, H, (width, height))```### Real-World Applications**Document Scanning:**- Detect document corners- Estimate homography- Dewarp to frontal view**Augmented Reality:**- Detect marker/object- Estimate pose- Overlay virtual content**Panoramas:**- Match overlapping images- Estimate transformations- Warp and blend**3D Reconstruction:**- Multiple views of scene- Estimate relative poses- Triangulate 3D points### Connection to Previous Labs**Lab 11 (Corners):**- Simple patch matching- Limited to translation- Not rotation/scale invariant**Lab 12 (today):**- SIFT features (rotation/scale invariant)- Full perspective transformations- RANSAC for robustness**Looking ahead:**- Structure from Motion (3D reconstruction)- Visual SLAM (robot navigation)- Deep learned features

### ✏️ Final ReflectionAnswer these questions:1. **Why is SIFT better than simple patch matching (Lab 11)?**2. **What does Lowe's ratio test accomplish?**3. **Why do we need RANSAC when estimating homography?**4. **How is a homography different from a rotation or scaling?**5. **What's one real-world application where homography would be useful?**

In [ ]:
# Your reflection:# 1. Why SIFT > patch matching?#    ...# 2. Lowe's ratio test purpose?#    ...# 3. Why RANSAC?#    ...# 4. Homography vs rotation/scaling?#    ...# 5. Real-world application?#    ...

---## 📋 Submission ChecklistBefore submitting, make sure:- [ ] Understood SIFT features and matching- [ ] Understood Lowe's ratio test- [ ] Understood RANSAC and homography estimation- [ ] Ran detection example successfully- [ ] Activity 1: Pose correction completed- [ ] Activity 2: Augmented reality completed- [ ] Activity 3: Panorama stitching completed- [ ] Final reflection questions answered- [ ] All cells executed in order**Grading (10 points):**| Component | Points ||-----------|--------|| Activity 1: Pose correction | 3 || Activity 2: Augmented reality | 3 || Activity 3: Panorama stitching | 3 || Final reflection | 1 |**Bonus (+1 point):** Implement automatic panorama stitching for 3+ images

---## 📝 Detailed Grading Rubric### Activity 1: Pose Correction (3 points)| Points | Criteria ||--------|----------|| 3.0 | Correct: inverts H, warps query image, correct output size || 2.5 | Works but minor issues (e.g., slight size mismatch) || 2.0 | Right idea but wrong direction or missing inversion || 1.0 | Attempted but major errors || 0.0 | Not attempted |### Activity 2: Augmented Reality (3 points)| Points | Criteria ||--------|----------|| 3.0 | Correct: other image replaces train in query, perspective matches || 2.5 | Works but simplified approach (assumes same size) || 2.0 | Partial: other image appears but wrong position/size || 1.0 | Attempted but significant errors || 0.0 | Not attempted |**This is the hardest activity** — accounting for size differences is tricky### Activity 3: Panorama Stitching (3 points)| Points | Criteria ||--------|----------|| 3.0 | Correct: both images visible, properly aligned || 2.5 | Works but slight misalignment || 2.0 | Both visible but poor alignment || 1.0 | Attempted but images not aligned || 0.0 | Not attempted |### Final Reflection (1 point)| Points | Criteria ||--------|----------|| 1.0 | All questions answered thoughtfully || 0.5 | Answered but superficial || 0.0 | Not attempted |---

---## 💡 Teaching Tips### Key Concepts to Emphasize1. **Homography = perspective transformation**   - 8 degrees of freedom   - Can represent rotation, scale, translation, AND perspective   - Needs at least 4 point correspondences2. **Forward vs inverse transformations**   - H: train → query   - H⁻¹: query → train   - Think carefully about direction!3. **RANSAC is essential**   - Even with ratio test, still have outliers   - RANSAC finds consensus among inliers   - Robust to 50%+ outliers!4. **SIFT vs patch matching**   - Patches: not rotation/scale invariant   - SIFT: handles these transformations   - 128-dim descriptors are distinctive### Common Student Confusions**"What's the difference between H and H⁻¹?"**→ Draw diagram showing direction of mapping→ H: object coords → scene coords→ H⁻¹: scene coords → object coords**"Why do we need RANSAC?"**→ Show: even after ratio test, some matches are wrong→ Outliers can completely break least-squares fitting→ RANSAC finds consistent majority**"How does warpPerspective work?"**→ For each pixel in output, compute source location using H→ Resample from source image→ This is "backward" mapping (prevents holes)### Demonstrations**Live demo:** Show panorama creation1. Take 2-3 overlapping photos2. Run through pipeline3. Show final stitched result**Interactive:** Let students try with their own images- Photos of documents- Overlapping phone photos- See what works/fails### If Students Struggle**Activity 1:**→ "Which direction? Query is distorted, train is correct"→ "What transformation undoes H?"→ Show: H × H⁻¹ = I (identity)**Activity 2:**→ "Start simple: assume same size, just use H"→ "For different sizes: need intermediate transformation"→ "Think: other → train → query"**Activity 3:**→ "Query already placed. Where should train go?"→ "H maps train → query. Need reverse"→ "Don't forget offset!"### Time Management**If running short:**- Activities 2-3 can be optional- Focus on Activity 1 (most important)- Demo others if time runs out**If students finish early:**- Try cv2.Stitcher on multiple images- Experiment with different transformations- Try on their own photos### Connection to Applications**Document scanning apps:**"Camscanner, Adobe Scan — same technique!"1. Detect document corners2. Compute homography  3. Warp to frontal view**AR (Pokémon GO, Snapchat):**"How they place virtual objects in real scenes"1. Detect markers/features2. Estimate camera pose3. Render virtual content with matching perspective**Google Street View panoramas:**"Automatically stitch images from car cameras"- Match features between frames- Estimate transformations- Blend into seamless panorama### Extension Topics- **Bundle adjustment:** Optimize multiple homographies together- **Blending:** Seamless panorama transitions- **3D reconstruction:** Multiple views → 3D structure- **Visual SLAM:** Robot localization using features---